# **Algorithmic Recourse: Practice**

**Hi, everyone.**

Tamara walked through every lesson of the module “Deep Dive into Counterfactuals”: she is 63, a former accountant, and she applied for an ML engineer position. The screening model invites a candidate to an interview when $p_{\text{invite}} \ge 0.5$, and it gave Tamara $p = 0.03$. Her application features are: age 63, 2 completed courses, 3 portfolio projects, 0.5 years of commercial experience. Age cannot be changed, and the model barely looks at it; courses, projects and experience are things Tamara can add.

In the lessons each method answered Tamara on its own. Here we gather the answers in one place and compare them on the properties from the step “Metrics and the main empirical fact” of the lesson “Implementations and conclusions”: validity, proximity, sparsity, plausibility, actionability, robustness.

In this practice you will:
- rebuild the data and the random-forest model from the lessons and check that Tamara gets $p = 0.03$;
- find the minimal change in the Wachter sense with a grid search and see how the weight $\lambda$ decides whether the counterfactual becomes valid;
- get three different plans from DiCE with age frozen and measure their proximity, sparsity and diversity;
- add a pull toward the prototype of the “invited” class and see the counterfactual become more typical and move further from Tamara;
- build FACE yourself (a k-NN graph, KDE density, Dijkstra) and compare a step-by-step path with the Wachter jump;
- set action costs in the spirit of Ustun, find the cheapest plan and a case where no recourse exists;
- check every plan you found on 15 retrained models and under execution noise, as ROAR and PRR do;
- put everything into one table of properties;
- repeat Growing Spheres from the lesson “Growing Spheres and DiCE” and see how the seed and the radius step change the point it finds;
- train a small conditional VAE and search for a counterfactual in its latent space, as REVISE and C-CHVAE do;
- set a causal model “courses → projects → experience” and find the cheapest causal recourse plan in the spirit of Karimi;
- write a genetic search in the spirit of GeCo and compare its number of model calls with exhaustive enumeration;
- rebuild the table of properties with every method of the module.

The notebook `PRACTICE_counterfactual_explanations` works with DiCE on another task (house price regression); this notebook is about the properties of recourse for one person.

**Happy coding!**

### Which libraries we use and why not the others

The module lessons name four tabular libraries. On Colab (Python 3.13, numpy 2.1.3) only one of them works, so we write the other methods ourselves in a few lines, following the formulas from the lessons.

| Library | What it offers | On Colab |
|---|---|---|
| `dice-ml` 0.12 | DiCE: genetic, random, kdtree | installs, we use it |
| `alibi` 0.9.6 | Wachter, `CounterfactualProto` | installing it downgrades numpy 2.1.3 → 1.26.4, Pillow → 10.4.0, transformers 5.18.0 → 4.57.6 and blis → 0.7.11 and breaks the rest of the environment; we write Wachter and prototypes ourselves as a grid search |
| CARLA (`carla-recourse` 0.0.5) | Growing Spheres, REVISE, C-CHVAE, FACE, ROAR, causal recourse | does not install: it needs tensorflow 1.14, which has no builds for Python 3.13 (plus numpy 1.19 and torch 1.7); we write FACE, the robustness check, Growing Spheres, REVISE, C-CHVAE and causal recourse ourselves (the VAE for REVISE and C-CHVAE in torch) |
| `GeCo.jl` | GeCo: genetic search with PLAF constraints | a Julia package that cannot be called from Python on Colab; we write a genetic search in the spirit of GeCo in numpy |
| OmniXAI 1.3.1 | a wrapper over DiCE and other explainers | installing it downgrades numpy to 1.23.5 and scikit-learn to 1.1.3 |
| `actionable-recourse` 1.0.1 | Ustun: integer programming | installs together with `cplex` and `mip==1.12`, but fails on the first `Flipset` call: it uses `np.float_`, which numpy 2 removed; besides, the method is built for a linear model, and Tamara's model is a forest. We compute action costs by enumerating every integer action |

All of this was checked by installing the packages into an environment with Colab's package versions as of 10.10.2026.

In [ ]:
!pip install dice-ml==0.12 -q

In [ ]:
import random, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
import dice_ml

pd.set_option('display.precision', 3)

## 1. Data and model

The code repeats the step “Growing Spheres: hyperparameters and limits” of the lesson “Growing Spheres and DiCE”: the same 2500 synthetic applications with seed 0, the same score (age does not enter it), the same model — standardization plus a random forest of 200 trees. We only wrapped the generation into a function `make_data(seed)`: section 7 needs it to retrain the model on new data, like the function `make(seed)` from the step “Robust recourse for Tamara” of the lesson “Ensuring stability: Robust Recourse (ROAR / PRR)”. With `seed=0` you get exactly the data from the lessons.

In [ ]:
cont = ["age", "courses", "projects", "exp"]
skill = ["courses", "projects", "exp"]           # what Tamara can change

def make_data(seed, n=2500):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "age":      rng.normal(42, 12, n).clip(22, 70),
        "courses":  rng.normal(4, 2, n).clip(0, 15),
        "projects": rng.normal(3, 2, n).clip(0, 20),
        "exp":      rng.normal(3, 2.5, n).clip(0, 20)})
    score = 0.5*df.courses + 0.7*df.projects + 0.6*df.exp + rng.normal(0, 1, n)  # age does NOT enter
    df["invited"] = (score > np.median(score)).astype(int)
    return df

def make_model(df):
    return Pipeline([
        ("sc", ColumnTransformer([("s", StandardScaler(), cont)])),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=0))
    ]).fit(df[cont], df["invited"])

df = make_data(0)
model = make_model(df)

tamara = np.array([63., 2., 3., 0.5])              # age, courses, projects, experience
def p(x, m=model): return m.predict_proba(pd.DataFrame([x], columns=cont))[0, 1]
def p_many(X, m=model): return m.predict_proba(pd.DataFrame(X, columns=cont))[:, 1]

print("p(Tamara is invited) =", round(p(tamara), 3))
print("share of invited in the data:", df.invited.mean())
df.head()

Let us check the promise “the model does not look at age”. Age is not in the score, but the forest can still split on it by fitting noise.

In [ ]:
imp = model.named_steps["clf"].feature_importances_
print("forest feature importances:", dict(zip(cont, imp.round(3).tolist())))
print("p at another age with the same skills:",
      {a: round(float(p([a, 2., 3., 0.5])), 3) for a in (25, 40, 63, 70)})

The forest gave age about 9% of the importance, yet Tamara's $p$ barely depends on age: from 0.025 to 0.05 across the whole range. So the answer “get younger” would not help Tamara even if it were possible — and every method below keeps age frozen at 63.

### How we measure the properties

We describe every counterfactual $x'$ we find with the same numbers (their definitions are in the step “Metrics and the main empirical fact” of the lesson “Implementations and conclusions”):

- **validity** — $p(x') \ge 0.5$ under the current model;
- **proximity** — the $L_1$ distance over skills, each skill divided by its MAD (median absolute deviation), as in Wachter; smaller means closer to Tamara;
- **sparsity** — how many of the three skills changed;
- **plausibility** — the mean distance to the 5 nearest real applications in standardized skills; smaller means more typical (for Tamara herself it is 0.177);
- **actionability** — age untouched and no skill decreased; for such plans we compute the cost in months of effort (section 6);
- **robustness** — the share of retrained models and the share of noisy executions under which the plan stays valid (section 7).

In [ ]:
mad = (df[skill] - df[skill].median()).abs().median().values
sd = df[skill].std().values
knn = NearestNeighbors(n_neighbors=5).fit(df[skill].values / sd)
unit_cost = np.array([1., 2., 12.])             # months per +1 course, +1 project, +1 year of experience (section 6)

def proximity(x):    return np.abs((x[1:] - tamara[1:]) / mad).sum()
def sparsity(x):     return int((np.abs(x[1:] - tamara[1:]) > 1e-9).sum())
def plausibility(x): return knn.kneighbors([x[1:] / sd])[0].mean()
def actionable(x):   return bool(x[0] == tamara[0] and (x[1:] >= tamara[1:] - 1e-9).all())
def cost(x):         return float((x[1:] - tamara[1:]) @ unit_cost)

def describe(x):
    return {"courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
            "p": round(p(x), 3), "valid": p(x) >= 0.5, "proximity": round(proximity(x), 2),
            "sparsity": sparsity(x), "plausibility": round(plausibility(x), 3),
            "actionable": actionable(x)}

found = {}                                       # counterfactuals of all methods go here
pd.DataFrame([describe(tamara)], index=["Tamara"])

## 2. Wachter: the minimal change

The basic formulation from the step “What do we already know about counterfactuals?” of the lesson “Introductory lesson: a reminder of counterfactual explanations”:

$$\arg\min_{x'}\ \max_{\lambda}\ \lambda\big(f(x') - y'\big)^2 + d(x, x'),$$

where $d$ is our proximity ($L_1$ scaled by MAD) and $y' = 1$. Alibi solves this with gradient descent (for a black box the gradient is estimated numerically), and the same search underlies `CounterfactualProto`. A forest has no gradient and there are only three skills, so we replace the descent with an exhaustive grid search with step 0.1: courses from 0 to 8, projects from 0 to 9, experience from 0 to 8 years, age 63. That is about 600 thousand points, and the forest scores them in a few seconds.

In [ ]:
t0 = time.time()
c, pr, e = np.meshgrid(np.arange(0, 8.01, 0.1), np.arange(0, 9.01, 0.1), np.arange(0, 8.01, 0.1), indexing="ij")
G = np.column_stack([np.full(c.size, 63.), c.ravel(), pr.ravel(), e.ravel()])
PG = p_many(G)
DG = (np.abs(G[:, 1:] - tamara[1:]) / mad).sum(1)       # proximity of every grid point
print(len(G), "points, %.1f s" % (time.time() - t0))

First let us see what the weight $\lambda$ does: for a fixed $\lambda$ we minimize $\lambda(p(x') - 1)^2 + d(x, x')$ over the whole grid.

In [ ]:
rows = []
for lam in [1, 10, 100, 1000]:
    k = np.argmin(lam * (PG - 1)**2 + DG)
    rows.append({"lambda": lam, **describe(G[k])})
pd.DataFrame(rows).set_index("lambda")

At $\lambda = 1$ the prediction penalty is cheaper than any step, and the optimum is Tamara herself with $p = 0.03$: the counterfactual is invalid. As $\lambda$ grows the point crosses the boundary, and at large $\lambda$ it goes deeper into the invited region, further from Tamara. That is why the formula has $\max_\lambda$: the weight is increased until the prediction becomes the desired one. In the limit the task is “the closest point among the valid ones”.

**Task 1.** Among the grid points with $p \ge 0.5$, find the point with the smallest proximity `DG` and store it in `x_wachter`.

*Self-check question:* which skills does the Wachter counterfactual change, what is its $p$ and how many skills does it touch? Compare with Growing Spheres from the step “Growing Spheres: hyperparameters and limits” of the lesson “Growing Spheres and DiCE” (courses $2\to2.2$, projects $3\to4.4$, experience $0.5\to1.6$).

<details><summary>Check yourself</summary>

The Wachter counterfactual: courses 2 (unchanged), projects $3 \to 4.4$, experience $0.5 \to 1.6$ years, $p = 0.505$, proximity 1.69; it changes 2 of the 3 skills. Growing Spheres in the lesson “Growing Spheres and DiCE” arrived at almost the same point (projects 4.4, experience 1.6) but also touched courses ($2 \to 2.2$): there the point was found by random sampling in a ball, while an exhaustive search with the $L_1$ distance leaves courses where they are. Solution: `x_wachter = G[np.argmin(np.where(PG >= 0.5, DG, np.inf))]`.
</details>

In [ ]:
x_wachter = ...  # Your code here
found["Wachter"] = x_wachter
pd.DataFrame([describe(x_wachter)], index=["Wachter"])

The Wachter counterfactual sits right on the boundary: $p$ is barely above 0.5. This is a property of the minimal solution, and in section 7 we will see what it costs.

## 3. DiCE: several different plans

First let us repeat the step “DiCE for Tamara” of the lesson “Growing Spheres and DiCE” as it is: genetic search, three counterfactuals, age excluded from `features_to_vary`, the query written with integers. In the lesson the search ran without a seed, so the third counterfactual changed from run to run; here we fix `random` and `numpy`, and all three plans reproduce. Next to the `invited` column printed by DiCE we put $p$ from the model itself.

In [ ]:
d = dice_ml.Data(dataframe=df, continuous_features=cont, outcome_name="invited")
m = dice_ml.Model(model=model, backend="sklearn")
query_int = pd.DataFrame([{"age": 63, "courses": 2, "projects": 3, "exp": 0.5}])   # as in the lesson

random.seed(0); np.random.seed(0)
cf_int = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query_int, total_CFs=3, desired_class=1, features_to_vary=skill)
out = cf_int.cf_examples_list[0].final_cfs_df.copy()
print(out.dtypes.to_dict())
out["model p"] = p_many(out[cont].values.astype(float)).round(3)
out

The first two plans match the lesson table: courses 3, projects 3, experience 3.2 and courses 2, projects 4, experience 1.6. DiCE prints `invited = 1` for them, yet the model gives $p = 0.285$ and $p = 0.255$: both plans are **invalid**. The cause is dtypes: the query stores courses and projects as integers, and DiCE casts the counterfactuals it found to the query dtypes, dropping the fractional part after the search. Inside the search there were, for example, 3.4 courses, the output kept 3, and the point fell back across the boundary. Hence the rule: the validity of a counterfactual is checked with the model's prediction, not with the column a library prints.

Let us write the query with floats and repeat the search.

In [ ]:
query = pd.DataFrame([{"age": 63., "courses": 2., "projects": 3., "exp": 0.5}])

random.seed(0); np.random.seed(0)
t0 = time.time()
cf_gen = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill)
print("genetic: %.2f s" % (time.time() - t0))
dice_gen = cf_gen.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
pd.DataFrame([describe(x) for x in dice_gen], index=["DiCE-1", "DiCE-2", "DiCE-3"])

Now all three plans are valid. The first two are the same paths as in the lesson, but without the dropped fractional part: DiCE-1 is 3.4 courses and 3.2 years of experience (the lesson kept 3 courses), DiCE-2 is 2.5 courses, 4.8 projects and 1.6 years of experience (the lesson kept 2 courses and 4 projects). The third plan is 5.5 courses with 2.9 projects: DiCE lowered the number of projects. `features_to_vary` froze age, but it does not restrict the direction of changes; such a plan is formally valid, yet it cannot be carried out (a portfolio project is not “undone”). DiCE has `permitted_range` for this.

For comparison, let us run the same query with the `random` method: it samples points in the permitted region and keeps the valid ones.

In [ ]:
random.seed(0); np.random.seed(0)
cf_rnd = dice_ml.Dice(d, m, method="random").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill, random_seed=0)
dice_rnd = cf_rnd.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
pd.DataFrame([describe(x) for x in dice_rnd], index=["random-1", "random-2", "random-3"])

**Task 2.** DiCE measures the diversity of a set with the determinant of a similarity matrix (the step “The formal statement of DiCE” of the lesson “Growing Spheres and DiCE”). Let us compute a simpler quantity: the mean pairwise $L_1$ distance scaled by MAD between the plans of a set. Complete the function `diversity(X)` for an array of plans `X` (rows are plans, columns are `cont`).

*Self-check question:* which method, `genetic` or `random`, has the larger mean proximity of its plans and the larger diversity of its set? What are the numbers?

<details><summary>Check yourself</summary>

`genetic`: mean proximity 2.55, diversity 3.42; `random`: mean proximity 8.19, diversity 10.82. The `random` method is further from Tamara and more diverse: it samples the whole permitted region and does not pull the plans toward the object. Solution: the mean over pairs $i < j$ of `np.abs((X[i, 1:] - X[j, 1:]) / mad).sum()`.
</details>

In [ ]:
def diversity(X):
    # Your code here
    ...

for name, X in [("genetic", dice_gen), ("random", dice_rnd)]:
    print("%-8s mean proximity = %.2f   mean sparsity = %.2f   diversity = %.2f"
          % (name, np.mean([proximity(x) for x in X]), np.mean([sparsity(x) for x in X]), diversity(X)))

Now let us forbid decreasing skills: `permitted_range` sets a corridor that starts at Tamara's current value.

In [ ]:
random.seed(0); np.random.seed(0)
cf_pr = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill,
    permitted_range={"courses": [2, 15], "projects": [3, 20], "exp": [0.5, 20]})
dice_pr = cf_pr.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
print("diversity = %.2f" % diversity(dice_pr))
pd.DataFrame([describe(x) for x in dice_pr], index=["DiCE+range-1", "DiCE+range-2", "DiCE+range-3"])

With the corridor all three plans are actionable, but they change only experience, and the diversity of the set dropped from 3.42 to 0.47. The actionability constraint was paid for with diversity: this is the first conflict between properties in the notebook.

In [ ]:
found["DiCE-1"], found["DiCE-2"], found["DiCE-3"] = dice_gen
found["DiCE+range"] = dice_pr[0]

## 4. Prototypes: a counterfactual close to typical invited candidates

We build the prototype as in the step “The prototype and the latent for Tamara” of the lesson “The problem of examples that do not exist: Prototypes and AE”: the mean of the 25 real invited applications closest to Tamara (distance in standardized features), with age kept at 63.

In [ ]:
sd_all = df[cont].std().values
inv = df[df.invited == 1][cont].values
proto = inv[np.argsort(np.linalg.norm((inv - tamara) / sd_all, axis=1))[:25]].mean(0)
proto[0] = 63                                   # age is immutable, keep it
pd.DataFrame([describe(proto)], index=["prototype"])

The numbers match the lesson (courses 3.3, projects 3.8, experience 2.2), and one thing deserves attention: the model does **not** invite the prototype itself, $p = 0.215$. The invited candidates closest to Tamara got their invitations partly thanks to the noise in the score, and their mean still lies before the boundary. So the prototype serves as a direction, not as an answer: it is used as a penalty inside the search, and the condition $p \ge 0.5$ takes care of validity.

Let us take the objective of the prototype method from the step “What is added to the objective” of the lesson “The problem of examples that do not exist: Prototypes and AE” without the autoencoder (the prototype itself partly plays its role):

$$L(x') = d(x, x') + \theta\,\Big\lVert \frac{x' - \text{proto}}{\text{sd}} \Big\rVert_2^2 \quad\text{subject to}\quad p(x') \ge 0.5.$$

**Task 3.** For each $\theta$ in the list, find the valid point of the grid from section 2 with the smallest $L$. The squared standardized distance to the prototype for every grid point is already computed in `DP`.

*Self-check question:* how do the proximity and the plausibility of the counterfactual change from $\theta = 0$ to $\theta = 10$?

<details><summary>Check yourself</summary>

At $\theta = 0$ (and at $\theta = 1$) the answer coincides with Wachter: proximity 1.69, plausibility 0.228. At $\theta = 10$: courses 3.5, projects 4.2, experience 2.5, proximity 3.16 (almost twice as large), plausibility 0.125 (smaller, that is more typical; Tamara herself has 0.177). Solution: `k = np.argmin(np.where(PG >= 0.5, DG + theta * DP, np.inf))`.
</details>

In [ ]:
DP = (((G[:, 1:] - proto[1:]) / sd)**2).sum(1)
rows, proto_cf = [], {}
for theta in [0, 1, 3, 10]:
    k = ...  # Your code here: index of the grid point
    proto_cf[theta] = G[k]
    rows.append({"theta": theta, **describe(G[k]), "to prototype": round(np.sqrt(DP[k]), 2)})
pd.DataFrame(rows).set_index("theta")

At small $\theta$ the prototype penalty is weaker than one grid step, and the answer coincides with Wachter. At $\theta = 10$ the counterfactual moves toward the prototype and lands in a denser region of applications, but its proximity grew from 1.69 to 3.16, while the plausibility distance dropped from 0.228 to 0.125. This is the trade-off of the lesson “The problem of examples that do not exist: Prototypes and AE”: plausibility is paid for with proximity.

In [ ]:
found["Prototype θ=10"] = proto_cf[10]

## 5. FACE by hand: a path through dense regions

CARLA with FACE does not install on Colab, so we repeat the code of the step “Three requirements for Tamara's recourse” of the lesson “Existence and reachability: FACE, Actionable and Causal Recourse”: the graph nodes are Tamara and the 2500 real applications (skills only), the edges connect 15 nearest neighbors in standardized skills, and the edge weight is $-\log \hat p\big(\tfrac{x_i + x_j}{2}\big)\cdot \lVert x_i - x_j \rVert$ with the KDE density at the edge midpoint. The target is an application that the model invites with $p \ge 0.6$ at Tamara's age and that itself lies in the denser half of the data. Dijkstra finds the path.

The only difference from the lesson: we compute the density at the edge midpoints with one KDE call for all edges at once instead of a loop. The numbers stay the same, and the cell runs several times faster.

In [ ]:
t0 = time.time()
X = df[skill].values.astype(float)
proba = model.predict_proba(df[cont].assign(age=63.))[:, 1]   # prediction for each application's skills at Tamara's age

pts = np.vstack([tamara[1:], X]); N = len(pts)  # Tamara is node 0
Z = (pts - X.mean(0)) / X.std(0)                # standardize for distances
kde = gaussian_kde(X.T)                         # data density

dist, idx = NearestNeighbors(n_neighbors=16).fit(Z).kneighbors(Z)
rows = np.repeat(np.arange(N), 15); cols = idx[:, 1:].ravel(); dd = dist[:, 1:].ravel()
dens = kde(((pts[rows] + pts[cols]) / 2).T)
w = np.maximum(-np.log(dens + 1e-12) * dd, 1e-6)  # -log(density) · distance
Gr = csr_matrix((w, (rows, cols)), shape=(N, N)); Gr = Gr.maximum(Gr.T)

dens_all = kde(X.T)
targets = np.where((proba >= 0.6) & (dens_all >= np.quantile(dens_all, 0.5)))[0] + 1
dpath, pred = dijkstra(Gr, indices=0, return_predecessors=True)
best = targets[np.argmin(dpath[targets])]

path = [best]
while path[-1] != 0: path.append(pred[path[-1]])
path = path[::-1]
print("%.1f s" % (time.time() - t0))
for k in path: print(np.round(pts[k], 2))
x_face = np.r_[63., pts[best]]
found["FACE"] = x_face
pd.DataFrame([describe(x_face)], index=["FACE"])

The path matches the lesson: Tamara $[2, 3, 0.5] \to [2, 3.55, 0.88] \to [2.18, 4, 1.51] \to [2.39, 4.37, 1.8]$, and the final application has $p = 0.625$ (the forest in `describe` gives the same number, since the endpoint is the skills of a real application at age 63). Every intermediate point is a real person from the sample.

Let us compare which areas the FACE path and the Wachter jump pass through. For each segment we take 50 points and look at the lowest KDE density on it; to make the number readable, we convert it into the share of real applications whose density is even lower.

In [ ]:
def min_density_share(points):
    ts = np.linspace(0, 1, 50)[:, None]
    low = min(kde((a + ts * (b - a)).T).min() for a, b in zip(points[:-1], points[1:]))
    return (dens_all < low).mean()

face_pts = pts[path]
jump_pts = np.vstack([tamara[1:], x_wachter[1:]])
for name, P in [("FACE", face_pts), ("Wachter", jump_pts)]:
    print("%-8s length in skills = %.2f, straight to the end = %.2f, "
          "share of applications with density below the sparsest point of the path = %.3f"
          % (name, sum(np.linalg.norm(b - a) for a, b in zip(P[:-1], P[1:])),
             np.linalg.norm(P[-1] - P[0]), min_density_share(P)))

In [ ]:
plt.figure(figsize=(7, 5))
sc = plt.scatter(X[:, 1], X[:, 2], c=proba, cmap="Blues", s=6, vmin=0, vmax=1)
plt.colorbar(sc, label="p(invited) at age 63")
plt.plot(face_pts[:, 1], face_pts[:, 2], "-o", color="#eb6834", lw=2, ms=8, label="FACE path")
plt.plot(jump_pts[:, 1], jump_pts[:, 2], "--s", color="#1baf7a", lw=2, ms=8, label="Wachter jump")
plt.scatter([tamara[2]], [tamara[3]], color="black", s=60, zorder=5, label="Tamara")
plt.xlim(1.5, 6.5); plt.ylim(0, 4)
plt.xlabel("projects"); plt.ylabel("experience, years")
plt.title("Applications, FACE path and Wachter jump (projected on projects and experience)")
plt.legend(loc="upper right"); plt.show()

In the figure the dots are real applications projected on projects and experience, and the color is the invitation probability at age 63. The orange polyline is the FACE path through real applications, the green dashed line is the Wachter jump straight to the boundary. Here both routes lie in a populated region: the sparsest point of the FACE path is denser than 66% of real applications, that of the Wachter jump denser than 55%. Tamara sits in the dense middle of the data, and there is no gap between her and the invited candidates; the FACE path goes through slightly denser places. FACE pays for a real endpoint and real intermediate steps with proximity: its endpoint is 1.93 away in skills along a straight line against 1.78 for Wachter, and its proximity is 2.07 against 1.69. If there were no dense trail, Dijkstra would find no path to the target applications — this is the FACE answer “no recourse”.

One more detail of the FACE endpoint. The target was selected by the condition $p \ge 0.6$, but that $p$ comes from a forest that saw this application during training. Let us retrain the model on the same data without this single application and ask again.

In [ ]:
model_wo = make_model(df.drop(index=best - 1))     # node best is row best - 1 of df
print("p of the lesson model = %.3f | p of the model without this application = %.3f"
      % (p(x_face), p(x_face, model_wo)))

Without its own application the forest gives the FACE endpoint $p = 0.35$, below the threshold. A random forest is overconfident on its training objects: each application landed in the bootstrap samples of most trees. So the confidence threshold $t_p$ of FACE should be checked with predictions made without that object (out-of-fold) — otherwise the “real invited person” turns out to be invited only by the model that memorized them. Section 7 will show this once more.

## 6. Action costs in the spirit of Ustun

Distance in feature space is geometry, while cost is Tamara's effort (the step “Actionable Recourse” of the lesson “Existence and reachability: FACE, Actionable and Causal Recourse”). Let us set rough prices in months: a course is 1 month, a project is 2 months, a year of experience is 12 months (it has to be lived through). The set of permitted actions $A(x)$ is integer: from 0 to 6 new courses, from 0 to 6 new projects and from 0 to 10 half-years of experience; skills cannot decrease and age cannot be touched. That is 539 actions, and an exhaustive enumeration over them replaces the integer programming of the `actionable-recourse` package: on a finite set it finds the exact minimum and proves just as exactly that no recourse exists.

**Task 4.** For every action in `acts`, compute its cost in months (`act_cost`) and find the cheapest valid action.

*Self-check question:* which plan is the cheapest, how many months does it cost and what is its $p$? How many months would the Wachter plan cost at the same prices?

<details><summary>Check yourself</summary>

The cheapest plan is four new courses (courses $2 \to 6$, projects and experience unchanged): 4 months, $p = 0.59$. The Wachter plan costs 16 months at the same prices (1.4 projects at 2 months each and 1.1 years of experience at 12 months each). Solution: `act_cost = acts @ np.array([1, 2, 6])` (a half-year of experience costs 6 months), `k = np.argmin(np.where(PA >= 0.5, act_cost, np.inf))`.
</details>

In [ ]:
acts = np.array(list(itertools.product(range(7), range(7), range(11))))   # +courses, +projects, +half-years of experience
XA = np.tile(tamara, (len(acts), 1))
XA[:, 1:] += acts * np.array([1., 1., 0.5])
PA = p_many(XA)
act_cost = ...  # Your code here: cost of every action in months
k = ...         # Your code here: index of the cheapest valid action
x_ustun = XA[k]
found["Ustun"] = x_ustun
print("cheapest plan:", acts[k], "→", x_ustun, "| cost =", act_cost[k],
      "months | p =", round(PA[k], 3))
print("cost of the Wachter plan at the same prices: %.1f months" % cost(x_wachter))

order = np.argsort(np.where(PA >= 0.5, act_cost, np.inf))[:6]
pd.DataFrame({"+courses": acts[order, 0], "+projects": acts[order, 1], "+exp": acts[order, 2] * 0.5,
              "months": act_cost[order], "p": PA[order].round(3)})

The cheapest plan looks nothing like the closest one. Wachter changed projects and experience: the model is more sensitive to them (in the score their weights are 0.7 and 0.6 against 0.5 for courses), and per unit of distance they raise $p$ more. In effort, however, courses are the cheapest. Proximity and actionability are different quantities, and the cost, not the geometry, decides between them.

Now let us narrow $A(x)$ to the real constraints of the next half-year: Tamara can manage at most two courses and two projects, and experience does not grow without a job.

In [ ]:
allowed = (acts[:, 0] <= 2) & (acts[:, 1] <= 2) & (acts[:, 2] == 0)
print("permitted actions:", allowed.sum(), "| valid among them:", (PA[allowed] >= 0.5).sum(),
      "| maximal p =", round(PA[allowed].max(), 3))

None of the 9 permitted actions gives $p \ge 0.5$: under these constraints no recourse exists for Tamara, and the enumeration proves it instead of assuming it. This is the kind of answer for which Ustun and co-authors formulated the task through a set of actions: “nothing available within half a year will help, experience is needed”.

## 7. Robustness in the spirit of ROAR and PRR

Recourse is a promise over time (the lesson “Ensuring stability: Robust Recourse (ROAR / PRR)”). Let us check every plan from two sides:

- **model shift (ROAR):** the company retrains the screening model on new data. Like the function `make(seed)` from the step “Robust recourse for Tamara”, we generate data with seeds 1–15 and train 15 models; the share of models with $p \ge 0.5$ is the robustness to the shift;
- **execution noise (PRR):** Tamara carries out the plan imprecisely. We add Gaussian noise to the skills with standard deviations of 0.5 courses, 0.5 projects and 0.7 years of experience (as in the lesson), 500 times; the share of valid executions is the robustness to noise.

Next to them we put the margin $p - 0.5$ and the proximity.

How this differs from the papers. In ROAR (Upadhyay et al., 2021) the set of models is a ball of perturbations of the parameters around the current model (for a nonlinear model, of its local linear LIME approximation), and the plan is found by the gradient of a minimax problem. A forest has no parameters that can be perturbed this way, so here the set of models is 15 forests retrained on new data, and the worst case at the end of the section is taken over them by enumeration. PRR (Pawelczyk et al., 2023) estimates the same share of invalid executions under Gaussian noise as `share_noise` does.

In [ ]:
t0 = time.time()
models = [make_model(make_data(s)) for s in range(1, 16)]
print("15 models in %.0f s" % (time.time() - t0))

**Task 5.** Complete `share_models(x)` — the share of the 15 retrained models in `models` with $p(x) \ge 0.5$. The noise function `share_noise(x)` is already written: it uses the same seed every time, so all plans are checked on the same perturbations.

*Self-check question:* what share of the retrained models invites Tamara under the Wachter plan and under the recipe “with a buffer” from the lesson “Ensuring stability: Robust Recourse (ROAR / PRR)” (courses 3, projects 4, experience 6.5)? What are their margins?

<details><summary>Check yourself</summary>

The Wachter plan is invited by 0 of the 15 retrained models (share 0.0), its margin is 0.005. The “buffer” plan is invited by 15 of 15 (share 1.0), its margin is 0.475. Solution: `return np.mean([p(x, mm) >= 0.5 for mm in models])`.
</details>

In [ ]:
def share_models(x):
    # Your code here
    ...

def share_noise(x, n=500):
    g = np.random.default_rng(3)                # execution noise on skills
    V = np.array(x, float) + g.normal(0, [0, 0.5, 0.5, 0.7], size=(n, 4))
    V[:, 1:] = np.clip(V[:, 1:], 0, None)
    return (p_many(V) >= 0.5).mean()

found["ROAR lesson: at boundary"] = np.array([63., 3., 4., 3.5])
found["ROAR lesson: buffer"] = np.array([63., 3., 4., 6.5])

true_thr = np.median(0.5*df.courses + 0.7*df.projects + 0.6*df.exp)   # score threshold without noise
def score_margin(x): return 0.5*x[1] + 0.7*x[2] + 0.6*x[3] - true_thr

rob = pd.DataFrame({name: {"p": round(p(x), 3), "margin": round(p(x) - 0.5, 3), "proximity": round(proximity(x), 2),
                           "score margin": round(score_margin(x), 2),
                           "share models": round(share_models(x), 3), "share noise": round(share_noise(x), 3)}
                    for name, x in found.items()}).T
rob

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, col, title in [(ax[0], "share models", "model shift (15 retrained)"),
                      (ax[1], "share noise", "execution noise (500 executions)")]:
    a.scatter(rob["margin"], rob[col], s=60, color="#2a78d6")
    for name, r in rob.iterrows():
        a.annotate(name, (r["margin"], r[col]), fontsize=8, xytext=(4, 3), textcoords="offset points")
    a.set_xlabel("margin = p − 0.5 under the current model"); a.set_ylabel("share valid")
    a.set_title(title); a.set_ylim(-0.05, 1.08); a.set_xlim(-0.03, 0.62)
plt.tight_layout(); plt.show()

In the two plots each dot is the plan of one method: horizontally its margin over the threshold under the current model, vertically the share of retrained models (left) and of noisy executions (right) under which the plan stays valid. The Wachter plan with a margin of 0.005 survives on none of the 15 retrained models and in only 19% of noisy executions; the “buffer” plan with a margin of 0.475 survives everywhere, but its proximity is 4.99 against 1.69 for Wachter.

The link between margin and robustness is rough, though: FACE has a margin of 0.125, yet the retrained models never invite it (its endpoint comes from the training sample, section 5), and the prototype plan at $\theta = 10$ has a margin of 0 and a share of models of 0.533. Our data is synthetic and we know the rule that labeled it, so the table has a “score margin” column — $0.5\cdot\text{courses} + 0.7\cdot\text{projects} + 0.6\cdot\text{experience}$ minus the median of this sum over the data. The shares of retrained models follow it almost exactly: plans with a negative score margin rely on a noise “pocket” of the current forest and disappear after retraining. A real task has no such rule, which is why ROAR and PRR check the perturbations themselves rather than the margin of a single model.

The last step is to build the robustness requirement into the search itself. On the integer actions from section 6 we find the cheapest plan that is valid **on all** 15 retrained models and on the current one (ROAR: the worst model of the set), and the cheapest plan that survives in at least 95% of noisy executions (PRR with $\delta = 0.05$).

In [ ]:
t0 = time.time()
worst = np.min([p_many(XA, mm) for mm in models + [model]], axis=0)     # worst model for every action
k_roar = np.argmin(np.where(worst >= 0.5, act_cost, np.inf))
cheap = np.argsort(np.where(PA >= 0.5, act_cost, np.inf))
k_prr = next(k for k in cheap if share_noise(XA[k]) >= 0.95)
found["ROAR (worst of 16)"] = XA[k_roar]
found["PRR (δ=0.05)"] = XA[k_prr]
for name, k in [("Ustun", np.argmin(np.where(PA >= 0.5, act_cost, np.inf))), ("ROAR", k_roar), ("PRR", k_prr)]:
    print("%-6s +%s → %s | cost = %d months | p = %.3f" % (name, acts[k], XA[k], act_cost[k], PA[k]))
print("%.1f s" % (time.time() - t0))

## 8. Summary table of properties

Let us put all the plans into one table. The arrow in a header shows which way is better: ↑ larger, ↓ smaller. The cost in months is computed only for actionable plans (age untouched, no skill decreased).

In [ ]:
summary = pd.DataFrame({name: {
    "courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
    "validity ↑": p(x) >= 0.5,
    "proximity ↓": round(proximity(x), 2),
    "sparsity ↓": sparsity(x),
    "plausibility ↓": round(plausibility(x), 3),
    "actionable ↑": actionable(x),
    "months ↓": round(cost(x), 1) if actionable(x) else np.nan,
    "share models ↑": round(share_models(x), 3),
    "share noise ↑": round(share_noise(x), 3)} for name, x in found.items()}).T
summary

The table confirms the main conclusion of the step “Metrics and the main empirical fact” of the lesson “Implementations and conclusions”: no method is the best in all columns at once.
- Wachter has the smallest proximity (1.69), but none of the 15 retrained models invites it, and under execution noise it is valid in 18.6% of cases.
- The prototype plan at $\theta = 10$ and FACE have the best plausibility (distances 0.125 and 0.129); yet no retrained model invites FACE, and the prototype plan costs 27.9 months.
- DiCE with the corridor and Ustun change the fewest skills (one each), but the first costs 40.8 months, and the second, the cheapest one (4 months), holds on only 1 of the 15 retrained models.
- DiCE-3 is valid but not actionable: it decreases the number of projects.
- Full robustness to model shift comes from the “buffer” plan (75 months) and from ROAR (8 months); ROAR, however, stays valid in a share of 0.668 of noisy executions, while PRR (also 8 months) is, conversely, valid in 99% of executions and on 11 of the 15 models.

The choice for Tamara depends on what matters more to her: the fastest is four courses from the Ustun plan, at the risk that they will not suffice after the screening model is retrained; the most reliable is the ROAR plan, two courses and three projects in 8 months, which withstands all 16 models.

## 9. Growing Spheres: search in growing spheres

In section 2 an exhaustive grid search found the closest valid point. In the lesson “Growing Spheres and DiCE” the same task was solved by Growing Spheres (Laugel et al., 2018), and here we repeat its code from the step “Growing Spheres: hyperparameters and limits”. The method needs only the model's predictions. It samples points in a ball around Tamara and grows the radius $r$ in steps of 0.05 until the ball contains at least one point that the model invites. Then it samples the ball of radius $r + 0.1$ densely, takes the invited point closest to Tamara in Euclidean distance and sparsifies it: one by one it returns skills to Tamara's values if the prediction does not change. CARLA, which implements the method, does not install on Colab (see the table at the start of the notebook), so the code is our own, in numpy. In Laugel et al. the search step works differently: the initial radius $\eta$ is first halved while the ball contains enemies, and then the points are drawn not in the whole ball but in the spherical layer between the previous and the new radius. The lesson code samples the whole ball again at every step, so it also checks the inner part where no enemies were found and spends extra model calls on it.

The lesson code is wrapped in the function `growing_spheres(seed)` with a counter of model calls `calls`; in section 12 we compare search methods by this counter.

**Task 6.** Complete the loop that grows the radius: increase `r` by `step` until at least one of the `m` points `shell(r, m)` is predicted as class 1, and stop at that radius.

*Self-check question:* at what radius did the method stop, which counterfactual did it find, and how many times did it call the model? How does it differ from the Wachter counterfactual of section 2?

<details><summary>Check yourself</summary>

Radius $r = 1.95$; the counterfactual: courses 2.19, projects 4.37, experience 1.64, $p = 0.505$, the same point as in the lesson “Growing Spheres and DiCE” (courses 2.2, projects 4.4, experience 1.6). Model calls: 216003, that is 39 balls of 4000 points, 60000 points in the layer it found and 3 checks during sparsification. The distance to Tamara is 1.79 in Euclidean distance over skills against 1.78 for Wachter, but proximity ($L_1$ scaled by MAD) is 1.83 against 1.69, and all three skills change, not two. Solution: inside `while r < 20:` the lines `cand = shell(r, m); cand = cand[pred(cand) == 1]`, then `if len(cand): break` and `r += step`.
</details>

In [ ]:
calls = {"n": 0}                                   # counter of model calls
def pred(X):
    calls["n"] += len(X)
    return (p_many(np.asarray(X, float)) > 0.5).astype(int)   # like model.predict: class 1 when p > 0.5

vary = [1, 2, 3]                                   # we change skills; age (0) is frozen

def growing_spheres(seed, step=0.05, m=4000, m_band=60000):
    gen = np.random.default_rng(seed)
    def shell(radius, m):                          # m points in a ball of radius `radius` around Tamara
        dirs = gen.normal(size=(m, len(vary)))
        dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)
        rad = gen.uniform(0, radius**len(vary), size=m) ** (1/len(vary))
        pts = np.tile(tamara, (m, 1))
        pts[:, vary] = np.clip(tamara[vary] + dirs*rad[:, None], 0, None)
        return pts
    r = step                                       # grow the radius until the ball contains a class-1 point
    while r < 20:
        ...  # Your code here: the ball shell(r, m), the check with pred, leaving the loop or growing r by step
    band = shell(r + 0.1, m_band); band = band[pred(band) == 1]        # sample the layer we found densely
    e = band[np.argmin(np.linalg.norm(band[:, vary] - tamara[vary], axis=1))].copy()   # the closest "enemy"
    for j in [vary[k] for k in np.argsort(np.abs(e[vary] - tamara[vary]))]:   # sparsify: drop unnecessary edits
        t = e.copy(); t[j] = tamara[j]
        if pred([t])[0] == 1: e = t
    return e, r

calls["n"] = 0
x_gs, r_gs = growing_spheres(2)
print("radius r = %.2f, model calls: %d" % (r_gs, calls["n"]))
print("Euclidean distance over skills: Growing Spheres %.2f, Wachter %.2f"
      % (np.linalg.norm(x_gs[1:] - tamara[1:]), np.linalg.norm(x_wachter[1:] - tamara[1:])))
found["Growing Spheres"] = x_gs
pd.DataFrame([describe(x_gs)], index=["Growing Spheres"])

Sparsification did not remove a single edit, and the reason is the norm. Growing Spheres looks for the closest point in a Euclidean ball, and the point of the boundary closest to Tamara in such a ball usually does not lie on an axis, so all skills change a little. Returning courses to 2 without moving anything else means stepping back from the boundary, and the prediction becomes “not invited” again. Wachter in section 2 searched by $L_1$, whose closest points lie on the axes more often, so courses stayed where they were.

The method is random, so let us check how the answer depends on the seed of the generator and on the radius step.

In [ ]:
rows = []
for seed in range(10):
    calls["n"] = 0
    x, r = growing_spheres(seed)
    rows.append({"seed": seed, "r": round(r, 2), **describe(x), "calls": calls["n"]})
gs_seeds = pd.DataFrame(rows).set_index("seed")
display(gs_seeds)

rows = []
for step in [0.02, 0.05, 0.2, 0.5]:
    calls["n"] = 0
    x, r = growing_spheres(2, step=step)
    rows.append({"radius step": step, "r": round(r, 2), **describe(x), "calls": calls["n"]})
pd.DataFrame(rows).set_index("radius step")

Over 10 seeds the stopping radius is between 1.85 and 1.95, projects between 4.37 and 4.41, experience between 1.55 and 1.64, and courses between 2.10 and 2.42: the direction of the answer is stable, and courses depend on which points got into the sample. All 10 plans are valid, all touch three skills, $p$ is between 0.505 and 0.565, proximity between 1.75 and 1.99.

The radius step trades precision for the number of calls. With step 0.02 the method made 440003 calls and found the point with courses 2.00, projects 4.44 and experience 1.60, proximity 1.72, almost the Wachter answer; with step 0.5 it made 76003 calls, and proximity is 1.90. By construction the method gives neither plausibility nor actionability: here skills only grow because no skill of Tamara's stands in the way of an invitation, not because the method checked it.

## 10. REVISE and C-CHVAE: search in the latent space of a VAE

The lesson “The problem of examples that do not exist: Prototypes and AE”, step “Continuing the idea of autoencoders”, proposes to search for a counterfactual not in feature space but in the latent space of a generative model: the decoder produces objects that look like the training data, so the point found lies on the data manifold. REVISE (Joshi et al., 2019) descends along the code $z$ by gradient:

$$z^{*} = \arg\min_{z}\ \mathrm{yloss}\big(f(g(z)), y'\big) + \lambda\, d\big(g(z), x\big),\qquad c = g(z^{*}).$$

C-CHVAE (Pawelczyk et al., 2020) takes a conditional VAE, encodes only the mutable features given the protected ones, and looks for the code closest to the object's code $z_0$ whose decoded object changes the prediction, using only the model's predictions:

$$z^{*}=\arg\min_{z}\ \lVert z-z_0\rVert\quad\text{subject to}\quad f\big(g(z, x_{\mathrm{prot}})\big)=y'.$$

Tamara has one protected feature, age. Let us train a small conditional VAE: the input is the three standardized skills, the condition is standardized age, the latent space is two-dimensional, 300 epochs of Adam. Training runs on CPU in a few seconds; the numbers of this section depend on training and may differ slightly on other hardware.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
mu_all, sd_all = df[cont].mean().values, df[cont].std().values
Xs = torch.tensor((df[cont].values - mu_all) / sd_all, dtype=torch.float32)   # standardized applications
S, A = Xs[:, 1:], Xs[:, :1]                        # skills (mutable) and age (condition)
y_t = torch.tensor(df.invited.values, dtype=torch.float32)

class CVAE(nn.Module):                             # conditional VAE: encodes skills given age
    def __init__(self, d=3, c=1, k=2, h=32):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d + c, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, 2 * k))
        self.dec = nn.Sequential(nn.Linear(k + c, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, d))
    def encode(self, s, c): return self.enc(torch.cat([s, c], 1)).chunk(2, 1)    # mean and log-variance
    def decode(self, z, c): return self.dec(torch.cat([z, c], 1))

vae = CVAE()
opt = torch.optim.Adam(vae.parameters(), 1e-3)
t0 = time.time()
for epoch in range(300):
    for b in torch.randperm(len(S)).split(128):
        mu, lv = vae.encode(S[b], A[b])
        z = mu + torch.randn_like(mu) * (0.5 * lv).exp()
        rec = ((vae.decode(z, A[b]) - S[b])**2).sum(1).mean()
        kl = (-0.5 * (1 + lv - mu**2 - lv.exp()).sum(1)).mean()
        loss = rec + 0.5 * kl
        opt.zero_grad(); loss.backward(); opt.step()
print("VAE in %.0f s; reconstruction error %.2f, KL %.2f" % (time.time() - t0, rec.item(), kl.item()))

def to_std(x): return torch.tensor(((np.asarray(x, float) - mu_all) / sd_all)[None], dtype=torch.float32)
def from_std(s, age=63.): return np.r_[age, s.detach().numpy()[0] * sd_all[1:] + mu_all[1:]]

xt = to_std(tamara)
with torch.no_grad():
    z0, _ = vae.encode(xt[:, 1:], xt[:, :1])
    x_rec = from_std(vae.decode(z0, xt[:, :1]))
pd.DataFrame([describe(tamara), describe(x_rec)], index=["Tamara", "Tamara after the VAE"])

The first thing to check in a latent method is how the VAE reconstructs Tamara herself. The decoder returned courses 2.48 and experience 1.5 instead of 2 and 0.5 years, and the model gives such an application $p = 0.255$ instead of 0.03. The reason is the data: the three skills are generated independently, they have no low-dimensional manifold, and a two-dimensional latent space has to compress three independent features. The search starts from the code $z_0$, and every counterfactual found in the latent space inherits this reconstruction error. This is what the condition “depends on the quality of the VAE” in the method card means.

REVISE needs the gradient of the classifier through the decoder, and a forest has no gradient. So for REVISE we train a differentiable surrogate, a small MLP on the same applications, and check the validity of the points found with the forest.

In [ ]:
torch.manual_seed(0)                               # REVISE needs a differentiable classifier: train an MLP on the same applications
mlp = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
opt = torch.optim.Adam(mlp.parameters(), 1e-2)
for epoch in range(300):
    loss = nn.functional.binary_cross_entropy_with_logits(mlp(Xs)[:, 0], y_t)
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    agree = ((mlp(Xs)[:, 0] > 0).numpy() == (model.predict(df[cont]) == 1)).mean()
    print("MLP: p(Tamara) = %.3f, agrees with the forest on %.1f%% of applications" % (torch.sigmoid(mlp(xt)).item(), 100 * agree))

mad_std = torch.tensor(mad / sd_all[1:], dtype=torch.float32)    # MAD in standardized units

def revise(lam, steps=500, lr=0.05):
    z = z0.clone().requires_grad_(True)
    opt = torch.optim.Adam([z], lr)
    for t in range(steps):
        s = vae.decode(z, xt[:, :1])               # decoded skills at Tamara's age
        logit = mlp(torch.cat([xt[:, :1], s], 1))[0, 0]
        loss = nn.functional.softplus(-logit) + lam * ((s - xt[:, 1:]).abs() / mad_std).sum()   # yloss + λ·d
        opt.zero_grad(); loss.backward(); opt.step()
    return from_std(vae.decode(z, xt[:, :1])), z.detach().numpy()[0]

revise_x, revise_z = {}, {}
for lam in [0.01, 0.1, 0.3, 1.0]:
    revise_x[lam], revise_z[lam] = revise(lam)
found["REVISE (λ=0.3)"] = revise_x[0.3]
pd.DataFrame([{"lambda": lam, **describe(x), "p of MLP": round(torch.sigmoid(mlp(to_std(x))).item(), 3)}
              for lam, x in revise_x.items()]).set_index("lambda")

The MLP agrees with the forest on 88% of applications. For all four values of $\lambda$ the forest invites the point found ($p$ from 0.51 to 0.905), but none of the points is actionable: courses drop from 2 to 0.06–1.29. The REVISE objective does not know that courses cannot be “unlearned”, and the decoder together with the MLP gradient takes the point to where the surrogate is confident. Plausibility did not work out either: the distance to the 5 nearest applications is from 0.258 to 0.463 against 0.177 for Tamara herself. At $\lambda = 1$ the surrogate and the forest disagree: the forest gives 0.51 and the MLP 0.472, so by its own classifier REVISE did not reach the boundary. For the summary table we take $\lambda = 0.3$, where the point is valid for both models.

C-CHVAE needs no gradient: it grows a ring around $z_0$ in the latent space, decodes the codes at age 63 and asks the forest for a prediction, that is, it runs Growing Spheres in the latent space.

**Task 7.** In the function `cchvae`, among the valid codes of the ring choose the one closest to $z_0$ and return its application `Xc[k]`, its code `Z[k]` and the radius `r`.

*Self-check question:* which plan did C-CHVAE find, and what is its plausibility compared with Growing Spheres and REVISE?

<details><summary>Check yourself</summary>

In the solved copy C-CHVAE stopped at radius 0.60 in the latent space and found courses 2.41, projects 4.34, experience 1.84 with $p = 0.500$ (exactly 100 of 200 trees; our validity condition is $p \ge 0.5$). Proximity is 2.08 against 1.83 for Growing Spheres, and the distance to the 5 nearest applications is 0.124, the smallest in the notebook (0.192 for Growing Spheres, 0.258 for REVISE with $\lambda = 0.3$, 0.129 for FACE). The numbers depend on training the VAE and may differ slightly on other hardware. Solution: `k = np.argmin(np.where(ok, np.linalg.norm(Z - z0.numpy(), axis=1), np.inf))`, then `return Xc[k], Z[k], r`.
</details>

In [ ]:
def decode_many(Z):                                # latent codes -> applications at age 63
    with torch.no_grad():
        Sd = vae.decode(torch.tensor(Z, dtype=torch.float32), xt[:, :1].repeat(len(Z), 1)).numpy()
    return np.column_stack([np.full(len(Z), 63.), Sd * sd_all[1:] + mu_all[1:]])

def cchvae(seed=0, step=0.05, m=2000):
    gen = np.random.default_rng(seed)
    r = step
    while r < 5:                                   # grow the ring r - step < |z - z0| <= r in the latent space
        d = gen.normal(size=(m, 2)); d /= np.linalg.norm(d, axis=1, keepdims=True)
        Z = z0.numpy() + d * gen.uniform(r - step, r, size=m)[:, None]
        Xc = decode_many(Z)
        ok = p_many(Xc) >= 0.5                     # black box: only the forest's predictions
        if ok.any():
            ...  # Your code here: index k of the valid code closest to z0 and return Xc[k], Z[k], r
        r += step

x_cchvae, z_cchvae, r_cchvae = cchvae()
print("radius in the latent space r = %.2f" % r_cchvae)
found["C-CHVAE"] = x_cchvae
pd.DataFrame([describe(x) for x in [x_gs, revise_x[0.3], x_cchvae]], index=["Growing Spheres", "REVISE (λ=0.3)", "C-CHVAE"])

Let us look at both methods in the latent space itself: color a two-dimensional lattice of codes by the forest's prediction for the decoded application and put the codes of real applications on it.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-3, 3, 121), np.linspace(-3, 3, 121))
Zg = np.column_stack([g1.ravel(), g2.ravel()])
Pg = p_many(decode_many(Zg)).reshape(g1.shape)     # the forest's prediction for every decoded latent point
with torch.no_grad():
    mu_data, _ = vae.encode(S, A)                  # codes of real applications

plt.figure(figsize=(7, 5.5))
cs = plt.contourf(g1, g2, Pg, levels=np.linspace(0, 1, 11), cmap="Blues")
plt.colorbar(cs, label="p(invited) at age 63")
plt.contour(g1, g2, Pg, levels=[0.5], colors="black", linewidths=1)
plt.scatter(mu_data[:, 0], mu_data[:, 1], s=2, color="gray", alpha=0.3, label="codes of real applications")
plt.scatter(*z0.numpy()[0], color="black", s=70, zorder=5, label="Tamara's code $z_0$")
plt.scatter(*revise_z[0.3], color="#eb6834", marker="s", s=70, zorder=5, label="REVISE (λ=0.3)")
plt.scatter(*z_cchvae, color="#1baf7a", marker="^", s=90, zorder=5, label="C-CHVAE")
plt.xlabel("$z_1$"); plt.ylabel("$z_2$"); plt.xlim(-3, 3); plt.ylim(-3, 3)
plt.title("VAE latent space: the forest's prediction for decoded applications")
plt.legend(loc="lower left", fontsize=8); plt.show()

In the figure the color is the forest's probability of an invitation for the application the decoder produces from a given code at age 63, the black line is the boundary $p = 0.5$, and the gray points are the codes of real applications. Tamara's code lies in a light area inside the cloud; C-CHVAE made a short step across the nearest boundary and stayed in the cloud, hence the small distance to real applications. REVISE followed the MLP gradient far beyond the edge of the cloud, into codes from which the decoder produces atypical applications. The decoder promises plausibility only where it has seen data; beyond the edge of the cloud of codes this promise does not hold.

## 11. Causal recourse: actions in a causal model

In the lesson “Existence and reachability: FACE, Actionable and Causal Recourse” (step “Causal Recourse (Karimi)”) recourse is searched for as an intervention in a structural causal model (SCM):

$$a^{*} = \arg\min_{a}\ \mathrm{cost}(a; x)\quad \text{subject to}\quad f\big(x^{SCF}_a\big) = y'.$$

The difference from Ustun in section 6 is $x^{SCF}_a$ instead of $x + a$: the final state is not assigned but computed by the SCM. The step “Three requirements for Tamara's recourse” says that Tamara's projects and experience are linked. Let us set an SCM by hand:

- courses $:= U_c$;
- projects $:= U_p + 0.5\cdot$courses (every second course ends with a portfolio project);
- experience $:= U_e + 0.25\cdot$projects (a project gives three months of experience).

The data generator of section 1 contains none of these links: the SCM describes how Tamara's actions propagate in the world, not how the applications were labeled. This is exactly what makes the method hard in practice: the SCM has to be set by hand, and there is usually no data to check it. CARLA, which implements the method, does not install on Colab, so we write Pearl's three steps ourselves:

1. **abduction**: recover the noise $U$ from Tamara's application;
2. **action**: a hard intervention $\mathrm{do}(X_i := x_i + \delta_i)$ replaces the equation of feature $X_i$ if $\delta_i > 0$;
3. **prediction**: recompute the features without intervention from their equations, top to bottom.

Actions and prices are the same as in section 6: 539 integer actions, a course costs one month, a project two months, half a year of experience six months. The price is paid for the action itself, and the shifts the SCM computes along causal links are free.

**Task 8.** In `scf`, complete projects and experience: under intervention, Tamara's value plus the increment; without intervention, the SCM equation from the already computed parent and the noise `U_tamara`.

*Self-check question:* which causal recourse plan is the cheapest, how much does it cost, and how does it differ from the Ustun plan?

<details><summary>Check yourself</summary>

The cheapest plan is three new courses for 3 months: by the SCM they give courses 5, projects 4.5 and experience 0.875 years, $p = 0.72$. The Ustun plan of section 6 is four courses for 4 months; by the SCM it gives $p = 0.92$, so it overpays by a month, because it does not know that courses bring projects. Without the SCM three courses would give $p = 0.395$. Solution: `pr = tamara[2] + delta[1] if do[1] else U_tamara[1] + b_cp * c` and `e = tamara[3] + delta[2] if do[2] else U_tamara[2] + b_pe * pr`.
</details>

In [ ]:
b_cp, b_pe = 0.5, 0.25                             # courses -> projects, projects -> experience (years per project)

def abduct(x):                                     # step 1, abduction: noise U from the observed application
    c, pr, e = x[1:]
    return np.array([c, pr - b_cp * c, e - b_pe * pr])

U_tamara = abduct(tamara)

def scf(delta, do):                                # steps 2 and 3: action do and prediction of descendants
    # delta: increments to courses, projects and experience; do: which features are intervened on
    c = tamara[1] + delta[0] if do[0] else U_tamara[0]
    pr = ...  # Your code here: projects
    e = ...   # Your code here: experience
    return np.array([63., c, pr, e])

D = acts * np.array([1., 1., 0.5])                 # the same 539 integer actions as in section 6
XS = np.array([scf(d, d > 0) for d in D])          # state after the action according to the SCM
PS = p_many(XS)
k_c = np.argmin(np.where(PS >= 0.5, act_cost, np.inf))
k_u = np.argmin(np.where(PA >= 0.5, act_cost, np.inf))
x_causal = XS[k_c]
found["Causal (SCM)"] = x_causal
print("Causal: action +%s, cost %d months, state after the action %s, p = %.3f"
      % (acts[k_c], act_cost[k_c], np.round(x_causal, 3), PS[k_c]))
print("Ustun:  action +%s, cost %d months; by the SCM the same action gives %s, p = %.3f"
      % (acts[k_u], act_cost[k_u], np.round(XS[k_u], 3), PS[k_u]))

order = np.argsort(np.where(PS >= 0.5, act_cost, np.inf))[:6]
pd.DataFrame({"+courses": acts[order, 0], "+projects": acts[order, 1], "+exp": acts[order, 2] * 0.5,
              "months": act_cost[order], "p by SCM": PS[order].round(3), "p without SCM": PA[order].round(3),
              "projects after": XS[order, 2].round(2), "exp after": XS[order, 3].round(3)})

The table shows the six cheapest plans by the SCM. The fourth row is unusual: the plan “+3 courses and +1 project” costs 5 months and gives 4.0 projects, fewer than “+3 courses” without a project (4.5). The hard intervention $\mathrm{do}(\text{projects} := 3 + 1)$ replaces the equation for projects entirely and cuts the arrow “courses → projects”, so the projects the new courses would have brought do not enter the result. This is a property of the lesson's formulation, not a bug in the code: an intervention that adds to the equation instead of replacing it has no such effect. Let us compare several plans in two worlds: without the SCM, as Ustun does, and by the SCM.

In [ ]:
def plan_row(name, dc, dp, de):
    d = np.array([dc, dp, de], float)
    x_naive = tamara + np.r_[0, d]
    x_scm = scf(d, d > 0)
    return {"plan": name,
            "projects: Ustun / SCM": f"{x_naive[2]:.2f} / {x_scm[2]:.2f}",
            "exp: Ustun / SCM": f"{x_naive[3]:.3f} / {x_scm[3]:.3f}",
            "p: Ustun / SCM": f"{p(x_naive):.3f} / {p(x_scm):.3f}"}

display(pd.DataFrame([plan_row("+3 courses", 3, 0, 0), plan_row("+3 courses and +1 project", 3, 1, 0),
                      plan_row("+2 projects", 0, 2, 0), plan_row("+2 projects and +1.5 years of experience", 0, 2, 1.5)]))

allowed = (acts[:, 0] <= 2) & (acts[:, 1] <= 2) & (acts[:, 2] == 0)
print("half-year constraints by the SCM: allowed actions %d, valid %d, maximum p = %.3f"
      % (allowed.sum(), (PS[allowed] >= 0.5).sum(), PS[allowed].max()))

By the SCM two projects give Tamara another half year of experience, but $p$ rises only from 0.05 to 0.115. In the plan “+2 projects and +1.5 years of experience” the intervention on experience cuts the arrow “projects → experience”, and the state coincides with the naive one: 2.0 years of experience, $p = 0.58$. Under the constraints of the next half year from section 6 there is no recourse by the SCM either: of 9 allowed actions 0 are valid, and the maximum $p$ is 0.285 against 0.205 without the SCM. Causal links made two courses more useful, but not enough.

## 12. GeCo: genetic search in interactive time

The lesson “Speeding up the search for real time: GeCo and CLUE”, step “GeCo — a fast search without gradients” (Schleich et al., 2021), promises a counterfactual in interactive time: a genetic search starts from candidates that change one feature, keeps feasibility constraints written in the PLAF language, selects the best, crosses and mutates them, and stops when the best candidate stops changing. The package `GeCo.jl` is written in Julia, so we write a search in the spirit of GeCo in numpy.

On the integer actions of section 6 exhaustive enumeration costs 539 model calls, and there is nothing to speed up. So let us take an action grid 10 times finer: steps of 0.1 course, 0.1 project and 0.1 year of experience, up to +12 courses, +12 projects and +10 years of experience, $121 \cdot 121 \cdot 101 = 1478741$ actions in total. A fractional course is part of a course; here the fine grid stands in for a large action space, the kind that appears when there are many features and many values. Our PLAF constraints are the same as in section 6: age does not change, skills do not decrease.

**Task 9.** Complete the fitness `fit`: for valid candidates ($p \ge 0.5$) it is the cost in months, for invalid ones $1000 + 100\cdot(0.5 - p)$, so that any valid candidate is better than any invalid one, and among invalid ones those closer to the boundary rank higher.

*Self-check question:* which plan did GeCo find with seed 0, in how many generations and model calls?

<details><summary>Check yourself</summary>

GeCo with seed 0: +3.2 courses (32 steps of 0.1), 3.2 months, $p = 0.510$, 19 generations and 418 model calls. The time in seconds depends on hardware, so it is more convenient to compare methods by the number of calls. Solution: `fit = np.where(pr >= 0.5, months, 1000 + (0.5 - pr) * 100)`.
</details>

In [ ]:
step_fine = np.array([0.1, 0.1, 0.1])              # a grid 10 times finer: +0.1 course, +0.1 project, +0.1 year
hi_fine = np.array([120, 120, 100])                # up to +12 courses, +12 projects, +10 years of experience
price_fine = step_fine * unit_cost                 # months per step for each skill

def p_actions(a):                                  # prediction for integer steps a, counts the calls
    calls["n"] += len(a)
    X = np.tile(tamara, (len(a), 1)); X[:, 1:] += a * step_fine
    return p_many(X)

def geco(seed, pop=40, keep=10, patience=10, max_gen=100):
    g = np.random.default_rng(seed)
    P = np.zeros((pop, 3), int)                    # GeCo start: candidates that change one feature
    j = g.integers(0, 3, pop); P[np.arange(pop), j] = g.integers(1, hi_fine[j] + 1)
    best, gen = [], 0
    while True:
        P = np.unique(P, axis=0)
        pr = p_actions(P)
        months = (P * price_fine).sum(1)
        fit = ...  # Your code here: months for valid ones, 1000 + (0.5 - p) * 100 for invalid ones
        order = np.argsort(fit); P, fit = P[order][:keep], fit[order][:keep]   # selection of the best
        best.append(fit[0]); gen += 1
        if gen >= max_gen or (len(best) > patience and best[-1] == best[-1 - patience]): break
        a, b = P[g.integers(0, len(P), pop)], P[g.integers(0, len(P), pop)]
        child = np.where(g.random((pop, 3)) < 0.5, a, b)                        # crossover
        mut = g.random((pop, 3)) < 0.3
        child = np.clip(child + mut * g.integers(-2, 3, (pop, 3)), 0, hi_fine)  # mutation; PLAF: skills do not decrease
        P = np.vstack([P, child])
    return P[0], fit[0], gen

calls["n"] = 0; t0 = time.time()
a_geco, m_geco, gen_geco = geco(0)
t_geco = time.time() - t0
x_geco = tamara + np.r_[0, a_geco * step_fine]
found["GeCo (grid 0.1)"] = x_geco
print("GeCo: +%s steps -> %s, %.1f months, p = %.3f; generations %d, model calls %d, %.2f s"
      % (a_geco, np.round(x_geco, 2), m_geco, p(x_geco), gen_geco, calls["n"], t_geco))

Let us compare with exhaustive enumeration of the same grid (one and a half million forest predictions, in blocks by the number of courses) and check how the GeCo answer depends on the stopping rule: `patience` is how many generations in a row the best candidate may stay the same.

In [ ]:
calls["n"] = 0; t0 = time.time()                   # exhaustive enumeration of the same fine grid, in blocks by courses
best_m, best_a = np.inf, None
for c0 in range(hi_fine[0] + 1):
    blk = np.array(list(itertools.product([c0], range(hi_fine[1] + 1), range(hi_fine[2] + 1))))
    pr = p_actions(blk); months = (blk * price_fine).sum(1)
    k = np.argmin(np.where(pr >= 0.5, months, np.inf))
    if pr[k] >= 0.5 and months[k] < best_m: best_m, best_a = months[k], blk[k]
t_full = time.time() - t0
print("exhaustive enumeration: +%s steps, %.1f months; model calls %d, %.1f s" % (best_a, best_m, calls["n"], t_full))

rows = []
for patience in [3, 10]:
    for seed in range(20):
        calls["n"] = 0
        a, m_, gen = geco(seed, patience=patience)
        rows.append({"patience": patience, "seed": seed, "months": round(m_, 1), "calls": calls["n"],
                     "found minimum": bool(abs(m_ - best_m) < 1e-9)})
geco_runs = pd.DataFrame(rows)
geco_runs.groupby("patience").agg(share_minimum=("found minimum", "mean"), median_months=("months", "median"),
                                  max_months=("months", "max"), median_calls=("calls", "median"))

Exhaustive enumeration found the same plan, +3.2 courses for 3.2 months, with 1478741 model calls; GeCo with seed 0 needed 418 calls, about 3500 times fewer. The time in seconds depends on hardware but grows with the number of calls: seconds for exhaustive enumeration, fractions of a second for GeCo, and for a model whose single prediction costs more than a forest's, the gap grows in the same proportion.

The stopping rule decides how far GeCo can be trusted. With `patience = 10` all 20 seeds found the minimum of 3.2 months, with a median of 517 calls. With `patience = 3` the search stops earlier (median 186 calls), 7 seeds of 20 found the minimum, the median cost is 3.85 months, and the worst seed gives 6.4 months. A genetic search does not prove that its plan is the cheapest and cannot prove that no recourse exists: only exhaustive enumeration over the set of actions, as in section 6, or an exact solver, as in Ustun's method, can do that.

CLUE from the same lesson looks for an object in which a model with uncertainty estimates (a Bayesian neural network) is confident, and it does not apply to Tamara's point-prediction forest, so it stays in the lesson “Speeding up the search for real time: GeCo and CLUE”. CLUE measures uncertainty by the entropy of the prediction; the cell below computes the entropy of Tamara's prediction: it is close to zero, the model confidently rejects her, and there is no doubt for CLUE to explain.

In [ ]:
# CLUE explains uncertainty: the entropy of the prediction in bits, H(p) = -p·log2(p) - (1-p)·log2(1-p)
def entropy_bits(q): return -(q * np.log2(q) + (1 - q) * np.log2(1 - q))
print("entropy of Tamara's prediction, bits:", round(entropy_bits(p(tamara)), 3))
print("entropy of a borderline application with p = 0.5, bits:", round(entropy_bits(0.5), 3))

## 13. Summary table with every method

Let us rebuild the table of section 8, adding the plans of sections 9–12. For the causal recourse plan the months column holds the cost of its action (three courses), not of the shift of the state: Tamara gets the projects and experience computed by the SCM without separate effort.

In [ ]:
effort = {name: cost(x) for name, x in found.items()}
effort["Causal (SCM)"] = float(act_cost[k_c])      # the effort of the causal plan is its action, not the shift of the state
summary_all = pd.DataFrame({name: {
    "courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
    "validity ↑": p(x) >= 0.5,
    "proximity ↓": round(proximity(x), 2),
    "sparsity ↓": sparsity(x),
    "plausibility ↓": round(plausibility(x), 3),
    "actionable ↑": actionable(x),
    "months ↓": round(effort[name], 1) if actionable(x) else np.nan,
    "share models ↑": round(share_models(x), 3),
    "share noise ↑": round(share_noise(x), 3)} for name, x in found.items()}).T
summary_all

The new rows confirm the conclusion of section 8 and add four observations to it.
- Growing Spheres and GeCo, like Wachter and Ustun, sit right at the boundary and survive on none of the 15 retrained models; GeCo also gives the cheapest actionable plan without a causal model, 3.2 months.
- C-CHVAE gives the smallest distance to real applications (0.124), but its $p = 0.500$ lies on the boundary, and the retrained models invite it in 1 case of 15.
- The causal recourse plan costs 3 months and holds on 11 of the 15 retrained models: by the SCM three courses bring another one and a half projects and 0.375 years of experience. This result is exactly as correct as the SCM we set ourselves.
- REVISE with $\lambda = 0.3$ holds on 11 of the 15 models, but it is not actionable: it decreases courses.

## **Conclusions**

In this practice we answered Tamara with every method of the module on the same data and the same model and saw that:
- the counterfactual with the smallest distance (Wachter) sits on the decision boundary and is fragile under model retraining and imprecise execution;
- validity has to be checked with the model's prediction: DiCE cast the plans to the integer dtypes of the query and printed `invited = 1` for invalid points;
- DiCE gives a menu of plans, but `features_to_vary` does not forbid decreasing skills; the `permitted_range` corridor makes the plans actionable and at the same time uniform;
- the prototype sets a direction toward typical invited candidates but may itself be invalid; pulling toward it raises plausibility at the cost of proximity;
- FACE leads through real applications and shows a step-by-step path, and pays for this with proximity; the confidence threshold for its target should be checked with a model trained without that application;
- action costs change the answer more than the distance metric does, and an exhaustive enumeration over the set of actions can prove that no recourse exists;
- a margin over the threshold buys robustness to model shift and execution noise, and pays for it with proximity and months;
- Growing Spheres arrives at almost the same point as Wachter, but it searches by Euclidean distance and therefore touches all three skills; the radius step sets its precision and its number of model calls;
- a search in the latent space of a VAE (C-CHVAE) gives the most typical plan of the notebook, but the answer is only as good as the VAE: a two-dimensional latent space reconstructs Tamara with an error, and REVISE, driven by the gradient of a surrogate MLP, leaves the cloud of real applications and decreases courses;
- a causal model changes both the cost and the plan itself: if courses bring projects and projects bring experience, three courses are enough for Tamara instead of four, and a hard intervention on projects cuts off what the courses contribute;
- a genetic search in the spirit of GeCo finds the same cheapest plan with hundreds of model calls instead of one and a half million, but it does not prove minimality and depends on its stopping rule;
- no method is the best on all properties at once: the choice depends on which property is more expensive to get wrong in your task.